# 01 · 原始 SEEG → 检测、时序、模板与统计（Fig1）

这个 notebook 从原始电压采样开始。每个计算 cell 标明面板；所有图由本次新检测的事件产生。默认输入是 Y1 的四个真实 200 秒片段，**并非论文的全患者 24 小时或 40 人队列**，因此不预设 18,190 个事件、TA/TB 数目或论文 p 值。

输入契约：未参考的 `samples_V[channel,sample]`、原始 `sfreq`、唯一通道名；EDF 也可直接读取。`config/raw_example.json` 显式给出双极配对、滤波、检测、打包参数。更换患者必须一起更换坏道、配对和坐标。Epilepsiae `.data/.head` 适配器保留在 `data_processing/src/preprocessing.py::load_epilepsiae_block`，其输出已参考，不能再次做双极。

方法顺序：原始 SEEG → 双极/重采样/陷波/带通 → BQK 检测 → 按患者规则选打包通道 → 群事件 → 拼接频谱 S³ 质心 → 参与通道 rank → masked K-means → 患者内与患者间统计。

In [ ]:
from pathlib import Path
import sys, os, json
# Only insert this independently copied package, never the original repository.
CODE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(CODE) not in sys.path: sys.path.insert(0, str(CODE))
from tools.runtime_guard import install_guard, local_import_report
install_guard()
import numpy as np
import matplotlib.pyplot as plt
from paper_paths import CODE_ROOT, INPUT_ROOT, OUTPUT_ROOT, sha256
from workflows.common import new_session, publish_session, previous_session, write_json
from plotting import computational as plot
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
from data_processing.raw_to_events import read_samples, preprocess, detect_groups, event_features, process_record, join_blocks
from analysis.from_events import cluster_events, patient_statistics, cohort_statistics
config = json.loads((CODE_ROOT/'config/raw_example.json').read_text())
raw_root = INPUT_ROOT/'computational'
OUT = new_session('fig1'); FIG = OUT/'figures'
print('配置:', config['scope']); print('输出:', OUT)


**Fig1A：原始波形和预处理。** 使用完整 200 秒片段滤波，再截事件；不能只对展示的 0.5 秒小窗滤波。这个透明入口使用 polyphase 重采样、Q=30 IIR 陷波和三阶 Butterworth；它与早期 legacy 的重采样实现不同，故不承诺逐点恢复历史图。几何采用原左触点别名，不能悄悄改成双极中点。

换成自己的EDF时，将 `config["records"]` 的 file 改成文件名，并明确 start_sec、duration_sec、block_id；无需先制作检测结果。可用 records[].channels 指定原始保留触点，preprocessing.drop_channels 排除坏触点。双极配对必须只包含保留触点，否则明确报错。

In [ ]:
record = config['records'][0]
samples, fs_native, native_names = read_samples(raw_root/record['file'], start_sec=record.get('start_sec',0), duration_sec=record['duration_sec'],channels=record.get('channels'))
pre = preprocess(samples, fs_native, native_names, config)
print('原始/处理后:', samples.shape, fs_native, pre['band_V'].shape, pre['sfreq'])


**Fig1B，步骤 1：单通道 HFO 与群事件。** BQK 的 80–250 Hz 子带检测按配置执行；打包通道由 mean + k·SD 的事件数门槛选择。群事件使用 ±30 ms 扩展、500 Hz 时间栅格、50% 打包通道和患者固定窗。这里保留当前 legacy pack 的两个细节：同一通道重叠检测可能重复计数；相互重叠的两个群窗一起剔除。

In [ ]:
detected = detect_groups(pre, config)
print('单通道事件数:', dict(zip(pre['names'], detected['counts'])))
print('打包通道:', detected['packing_channels'], '群事件:', len(detected['windows']))
plot.save(plot.waveform(pre, detected), FIG, 'fig1_A_waveforms', '从本次预处理后信号截取的第一个群事件。用于检查原始电压、滤波和通道顺序。')
plt.show()


**Fig1B，步骤 2：序列如何得到。** 先拼接本段所有新群窗，计算 50 ms Hamming / 40 ms overlap 幅度谱，每个通道独立作 σ=1.5 Gaussian 平滑，再取 50<f<300 Hz。整个事件窗内以 S³/ΣS³ 算时间质心；图中的竖线来自计算，没有对齐峰值或人工移位。

In [ ]:
features = event_features(pre, detected)
features['block_ids'] = np.full(features['ranks'].shape[1], record['block_id'])
plot.save(plot.spectra(pre, features, index=0), FIG, 'fig1_B_centroids', '本次检测的第一个群事件及完整事件窗 S³ 时间质心。按参与通道显示，缺席通道不填 rank。')
plt.show()


**Fig1C：从质心到 rank，并汇总记录。** 参与 mask 来自检测区间与群窗的重叠；未参与通道保留缺失。跨记录按名称对齐，不能按 NPZ 行号拼接。同一 recording 的多个片段必须使用同一 block_id。

In [ ]:
blocks = [features]
for record in config['records'][1:]:
    _, _, block = process_record(record, config, raw_root)
    blocks.append(block)
    print(record['block_id'], block['ranks'].shape[1], 'new group events')
events = join_blocks(blocks)
plot.save(plot.rank_heatmap(events), FIG, 'fig1_C_ranks', '全部示例片段重新计算的群事件 rank 矩阵。色标仅表示本事件内先后，空白为未参与。')
print('汇总:', events['ranks'].shape); plt.show()


**Fig1E：K-means 如何得到模板。** 先仅在参与通道内重排、归一化 rank，再对缺失位置作 event-median=0.5 插补。K=2 是这一图的方法设定，不是生理状态数目的证据；cluster label 按事件数固定排序，TA/TB 不自动对应 SOZ 或状态。

In [ ]:
clustered = cluster_events(events, k=2, seed=0)
print('新拟合标签数:', np.unique(clustered['labels'], return_counts=True))
plot.save(plot.rank_heatmap(events, clustered), FIG, 'fig1_E_kmeans', '本次 masked-rank K-means 聚类后的矩阵。模板从本次事件重新估计，不读取旧标签。')
plt.show()


**Fig1D：如何计算统计。** Matching index 比较事件与患者模板的通道对先后，只计共同参与通道。置换保持参与 mask。先得到每位患者的观察值、null 和模板内值，再对患者做配对检验。默认一个患者的四段记录只算 **一个统计单位**，不足以重现队列显著性。

In [ ]:
stats = patient_statistics(events, clustered, permutations=200, seed=0)
cohort = cohort_statistics({config['subject']: stats})
print({k:v for k,v in cohort.items() if not isinstance(v,np.ndarray)})
plot.save(plot.statistics(stats,panel="D"), FIG, 'fig1_D_statistics', '从新事件计算观察值、置换基线及两个模板内 matching index。当前只有一个患者，不能据此报告论文队列的显著性。')
plt.show()


**Fig1F：模板内时序一致性。** 对本次K-means的各模板内事件计算匹配度，与全事件统计并列。当前只有一个患者，不能把两类事件当成两个独立患者。

In [ ]:
plot.save(plot.statistics(stats,panel="F"), FIG, "fig1_F_template_statistics", "本次新事件的模板内与总体时序一致性。完整队列按患者汇总。"); plt.show()


**交给第二本 notebook 的计算接口。** 只保存本次计算产生的事件与坐标；这不是读取论文图件。完整队列请用各患者的 raw 配置运行 `workflows/cohort.py`，它重新处理每名患者，并以患者为单位合并统计。

In [ ]:
coordinates = json.loads((raw_root/config['coordinates']).read_text())
xyz = np.array([coordinates.get(n,[np.nan]*3) for n in events['channel_names']])
np.savez_compressed(OUT/'events.npz', **events, coords_mm=xyz)
write_json(OUT/'statistics.json', stats)
report = dict(scope=config['scope'], n_events=events['ranks'].shape[1], n_patients=1,
              input_sha256={r['file']:sha256(raw_root/r['file']) for r in config['records']},
              configuration=config, imports=local_import_report())
publish_session('fig1', OUT, report)
print('Notebook 2 将读取本次计算:', OUT/'events.npz')
